# 03 — EDA and data-quality audit

Understand balance, rendering quality, image properties, and potential problems before selecting a model.

# Setup

Run this notebook from the repository root. In Google Colab, clone the GitHub repository first and replace the placeholder URL.

In [ ]:
from pathlib import Path
import os, sys

REPO_URL = "PASTE_YOUR_GITHUB_REPOSITORY_URL_HERE"
if 'google.colab' in sys.modules:
    if not Path('/content/fontsense-capstone').exists():
        if 'PASTE_' in REPO_URL:
            raise ValueError('Replace REPO_URL with your GitHub repository URL first.')
        !git clone {REPO_URL} /content/fontsense-capstone
    os.chdir('/content/fontsense-capstone')
    %pip install -q -r requirements.txt
    %pip install -q -e .
else:
    root = Path.cwd()
    if root.name == 'notebooks':
        root = root.parent
    os.chdir(root)
    os.environ['PYTHONPATH'] = str(root / 'src') + os.pathsep + os.environ.get('PYTHONPATH', '')
    if str(root / 'src') not in sys.path:
        sys.path.insert(0, str(root / 'src'))
print('Project root:', Path.cwd())


In [ ]:
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
MANIFEST = 'data/processed/fontsense_google/manifest.csv'
manifest = pd.read_csv(MANIFEST)

In [ ]:
display(manifest.groupby(['split','category']).size().unstack(fill_value=0))
display(manifest.groupby(['split','category'])['family'].nunique().unstack(fill_value=0))

In [ ]:
counts = manifest.groupby(['split','category']).size().unstack(fill_value=0)
counts.plot(kind='bar', figsize=(10,5), title='Images by split and class')
plt.ylabel('Images'); plt.tight_layout()

In [ ]:
# Validate files and dimensions
checks=[]
for path in manifest.image_path:
    try:
        with Image.open(path) as im:
            checks.append((path, im.width, im.height, im.mode, True, ''))
    except Exception as exc:
        checks.append((path, None, None, None, False, str(exc)))
quality = pd.DataFrame(checks, columns=['path','width','height','mode','readable','error'])
display(quality.describe(include='all'))
assert quality.readable.all()

## Written observations

Add short conclusions after every important table or chart. Discuss class balance, independent family counts, label ambiguity, and the synthetic-to-real domain gap.